# Insight 2.0: Vital Status Prediction

**Frozen benchmark:** `submission6.csv` is the previously scored Submission 6 artifact. Its recorded recipe is 55% v5 (without target encoding) + 45% v4 (with fold-safe target encoding), pseudo-labels at probabilities >=0.95 or <=0.05, a 50/50 blend of the original and pseudo-trained predictions, and threshold `0.684` (30,411 of 36,000 predictions labelled Dead).

**Pending candidate:** `submission.csv` is the unscored candidate formed from 80% `archive/probs_v6_final.npy` and 20% `archive/probs_nn.npy`, followed by a deterministic top-30,411 decision. It preserves Submission 6's Dead count and changes 292 labels. No leaderboard result is claimed for this candidate.

This living notebook keeps the frozen benchmark separate from the pending candidate. The training cell delegates to `pipeline_v6.py`; until a clean full run is checked against `submission6.csv`, the frozen CSV--not a fresh rerun--is the exact benchmark artifact. Dead is the positive class and model selection uses F1.

## EDA
Load the data, verify schema, class balance, missingness, and train/test category coverage before fitting anything.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

ROOT = Path.cwd().resolve()
required_paths = [ROOT / name for name in ('train.csv', 'test.csv', 'pipeline_v6.py')]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f'Run this notebook from the project directory; missing: {missing_paths}')
train_df = pd.read_csv(ROOT / 'train.csv')
test_df = pd.read_csv(ROOT / 'test.csv')
assert {'patient_id', 'vital_status'} <= set(train_df.columns)
assert 'patient_id' in test_df.columns and 'vital_status' not in test_df.columns
assert train_df['patient_id'].is_unique and test_df['patient_id'].is_unique
assert set(train_df['vital_status'].dropna().unique()) <= {'Dead', 'Alive'}
y = (train_df['vital_status'] == 'Dead').astype('int8')
print('train:', train_df.shape, 'test:', test_df.shape)
print('dead rate:', f'{y.mean():.3%}')
display(train_df.head())
display(train_df.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_rate'))

## Preprocessing
Categorical fields are ordinal encoded with an unknown-value sentinel. Numeric missing values are replaced only at the final model matrix boundary. Target encoding is computed inside each training fold, with an inner split for training rows.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['age_recode','race','sex','origin','primary_site','marital_status_at_diagnosis',
            'sequence_number','site_recode_icdo3_who2008','grade_recode_thru2017','laterality',
            'diagnostic_confirmation','summary_stage','derived_eod2018t_recode2018',
            'derived_eod2018n_recode2018','derived_eod2018m_recode2018',
            'seer_combined_metsatdxbone2010','seer_combined_metsatdxbrain2010',
            'seer_combined_metsatdxliver2010','seer_combined_metsatdxlung2010',
            'rx_summ_surgprim_site19982022','rx_summ_surgprim_site20232023',
            'rx_summ_scope_reglnsur2003','rx_summ_surgothregdis2003','rx_summ_surgradseq',
            'reason_nocancer_directed_surgery','radiation_recode','tumor_size_overtime',
            'tumor_size_summary','cs_tumor_size20042015','cs_extension20042015']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(train_df[cat_cols].fillna('__NA__').astype(str))
coverage = []
for column in cat_cols:
    train_values = set(train_df[column].fillna('__NA__').astype(str))
    test_values = set(test_df[column].fillna('__NA__').astype(str))
    coverage.append({'column': column, 'test-only categories': len(test_values - train_values)})
print('categorical columns:', len(cat_cols))
display(pd.DataFrame(coverage).sort_values('test-only categories', ascending=False).head(10))

## Feature Engineering
The v6 feature builder creates age, TNM/stage order, node-ratio, metastasis, treatment, missingness, interaction, and target-free frequency features. The cell below loads the maintained implementation to avoid silently duplicating a different recipe. The final review notebook should inline or package this dependency with an integrity check.

In [ ]:
# Keep the canonical implementation in one maintained source file.
# This cell is intentionally explicit so the notebook and script use the same features.
import ast
v6_source = (ROOT / 'pipeline_v6.py').read_text()
tree = ast.parse(v6_source)
feature_node = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == 'build_features')
feature_code = ast.get_source_segment(v6_source, feature_node)
LEAKAGE_CLEAN = False  # Submission 6 setting required by build_features().
exec(feature_code, globals())
X_train_num = build_features(train_df)
X_test_num = build_features(test_df)
print('numeric/derived features:', X_train_num.shape[1])

## Model Development
The recorded Submission 6 recipe trains v5 and v4 LightGBM, XGBoost, and CatBoost models over 3 repeats x 5 folds, selects the v5/v4 blend from OOF predictions, pseudo-labels test rows at the 95%/5% confidence bounds, retrains six models over 5 folds, and averages original and pseudo-trained predictions 50/50. The frozen run selected 55% v5 + 45% v4.

In [ ]:
# Full v6 training is intentionally opt-in because it fits 120 boosted-tree models.
# The current script reselects the blend and rate-based threshold, writes only under
# artifacts/v6_rerun/, and never overwrites the frozen or pending root submissions.
# Verify any rerun against Submission 6 before calling it an exact reproduction.
RUN_FULL_V6 = False
if RUN_FULL_V6:
    exec(compile(v6_source, 'pipeline_v6.py', 'exec'), {'__name__': '__main__'})
else:
    print('Set RUN_FULL_V6=True to run v6 into artifacts/v6_rerun/.')

## Frozen Model Parameters
The fixed v4/v5 model parameter dictionaries in `pipeline_v6.py` are treated as part of the recorded Submission 6 recipe. Any later tuning is a separate experiment and must be nested inside training folds and judged by held-out F1, not by public-leaderboard score.

In [ ]:
# Threshold tuning is a separate validation decision.
def best_f1_threshold(probabilities, labels, grid=np.linspace(0.2, 0.8, 6001)):
    scores = np.array([f1_score(labels, probabilities >= threshold) for threshold in grid])
    index = int(scores.argmax())
    return float(grid[index]), float(scores[index])

print('Use fold-held-out probabilities here; never tune on test labels.')

## Evaluation
`oof_step1.npy` is the cached OOF prediction for the original v4/v5 blend generated by `step1_cv_diagnostic.py`; it is **pre-pseudo-labeling**, not an OOF estimate of the final Submission 6 ensemble. The full-OOF threshold sweep below is diagnostic and slightly optimistic for threshold selection; final selection requires an outer held-out or cross-fitted threshold decision.

In [ ]:
from sklearn.calibration import calibration_curve
assert (ROOT / 'oof_step1.npy').is_file() and (ROOT / 'y_step1.npy').is_file()
oof = np.load(ROOT / 'oof_step1.npy')
cached_y = np.load(ROOT / 'y_step1.npy')
assert len(oof) == len(y) and np.array_equal(cached_y, y.to_numpy())
threshold, score = best_f1_threshold(oof, y.to_numpy())
print(f'Pre-pseudo v4/v5 OOF F1={score:.6f}; threshold={threshold:.4f}; predicted Dead rate={(oof >= threshold).mean():.3%}')
prob_true, prob_pred = calibration_curve(y, oof, n_bins=10, strategy='quantile')
display(pd.DataFrame({'predicted': prob_pred, 'observed': prob_true, 'gap': prob_true - prob_pred}))

## Prediction
`submission6.csv` is the immutable, previously scored reference and must not be overwritten. The canonical pending candidate is `submission.csv`: blend the exact archived probability vectors as `0.80 * archive/probs_v6_final.npy + 0.20 * archive/probs_nn.npy`, rank the rows with a stable sort, and label exactly the top 30,411 rows as Dead. This isolates ranking changes while preserving the frozen benchmark's class count.

The validation cell checks both archived probability checksums, reconstructs the frozen reference, constructs the expected candidate in memory, and requires the current `submission.csv` to match it exactly. It does not retrain models or overwrite either CSV, and it does not claim a leaderboard result for the pending candidate.

In [ ]:
import hashlib

EXPECTED_COLUMNS = ['patient_id', 'vital_status']
EXPECTED_ROWS = 36_000
EXPECTED_DEAD = 30_411

# Frozen, previously scored reference.
reference_path = ROOT / 'submission6.csv'
assert reference_path.is_file(), 'Frozen Submission 6 artifact is missing.'
reference = pd.read_csv(reference_path)
assert list(reference.columns) == EXPECTED_COLUMNS
assert len(reference) == len(test_df) == EXPECTED_ROWS
assert reference['patient_id'].is_unique
assert reference['patient_id'].equals(test_df['patient_id'])
assert set(reference['vital_status']) <= {'Dead', 'Alive'}
assert int(reference['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD
reference_sha256 = hashlib.sha256(reference_path.read_bytes()).hexdigest()
assert reference_sha256 == 'fd7cca1ee4a7654757adb78934baf42a07ae264dc581217df3e7863b552ef477', (
    'Submission 6 no longer matches the frozen artifact.'
)

# Exact archived probability inputs for the pending candidate.
frozen_probability_path = ROOT / 'archive' / 'probs_v6_final.npy'
nn_probability_path = ROOT / 'archive' / 'probs_nn.npy'
assert frozen_probability_path.is_file(), 'Frozen Submission 6 probabilities are missing.'
assert nn_probability_path.is_file(), 'Archived NN probabilities are missing.'
frozen_probability_sha256 = hashlib.sha256(frozen_probability_path.read_bytes()).hexdigest()
nn_probability_sha256 = hashlib.sha256(nn_probability_path.read_bytes()).hexdigest()
assert frozen_probability_sha256 == 'aca54c31462449df432e1edda5da81a6d04e242c8985cfde0e5983c6d0d92ab6'
assert nn_probability_sha256 == '7ec4721ae7d4eccb35ebc5821014e581ad2da4e872775d8a4845f37423b1ce46'
frozen_probabilities = np.load(frozen_probability_path)
nn_probabilities = np.load(nn_probability_path)
assert frozen_probabilities.shape == nn_probabilities.shape == (EXPECTED_ROWS,)
assert np.isfinite(frozen_probabilities).all() and np.isfinite(nn_probabilities).all()
reconstructed = np.where(frozen_probabilities >= 0.684, 'Dead', 'Alive')
assert np.array_equal(reconstructed, reference['vital_status'].to_numpy())

def deterministic_top_k_labels(probabilities, positive_count):
    # Stable sorting makes row order the deterministic tie-breaker.
    order = np.argsort(probabilities, kind='mergesort')
    labels = np.full(len(probabilities), 'Alive', dtype='<U5')
    labels[order[-positive_count:]] = 'Dead'
    return labels

candidate_probabilities = 0.80 * frozen_probabilities + 0.20 * nn_probabilities
expected_labels = deterministic_top_k_labels(candidate_probabilities, EXPECTED_DEAD)
expected_candidate = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'vital_status': expected_labels,
})
changed_vs_reference = expected_labels != reference['vital_status'].to_numpy()
assert int(changed_vs_reference.sum()) == 292
assert int(expected_candidate['vital_status'].eq('Dead').sum()) == EXPECTED_DEAD

# Validate the already-generated pending file; do not overwrite it here.
candidate_path = ROOT / 'submission.csv'
assert candidate_path.is_file(), 'Pending submission.csv is missing.'
candidate = pd.read_csv(candidate_path)
assert list(candidate.columns) == EXPECTED_COLUMNS
assert len(candidate) == EXPECTED_ROWS and candidate['patient_id'].is_unique
assert candidate['patient_id'].equals(expected_candidate['patient_id'])
assert set(candidate['vital_status']) <= {'Dead', 'Alive'}
assert np.array_equal(candidate['vital_status'].to_numpy(), expected_labels), (
    'submission.csv does not match the canonical 80/20 top-30,411 construction.'
)
candidate_sha256 = hashlib.sha256(candidate_path.read_bytes()).hexdigest()

print(f'Frozen Submission 6 verified: {EXPECTED_ROWS:,} rows, {EXPECTED_DEAD:,} Dead; sha256={reference_sha256}')
print(f'Pending submission.csv verified: 80% frozen v6 + 20% NN, {EXPECTED_DEAD:,} Dead, 292 changes; sha256={candidate_sha256}')
print('The pending candidate is unscored; no leaderboard result is asserted.')